# 1. Konfigurasi & Load Environment Variables

In [19]:
from pathlib import Path
from markitdown import MarkItDown
from kaggle_secrets import UserSecretsClient
from groq import Groq


user_secrets = UserSecretsClient()
GROQ_API_KEY = user_secrets.get_secret("GROQ_API_KEY")
groq_client = Groq(api_key=GROQ_API_KEY)

# Folder input dan output
input_folder = Path("/kaggle/input/datasets/brianmaxwellketaren/skripsi-rag-pdf")   # ganti dengan path folder Anda
output_folder = Path("output_md")
output_folder.mkdir(exist_ok=True)

# Inisialisasi converter
md_teks = MarkItDown()
md_gambar = MarkItDown(
    enable_plugins=True,
    llm_client=groq_client,
    llm_model="meta-llama/llama-4-scout-17b-16e-instruct",
)

# Ekstensi yang ingin dikonversi (bisa ditambah sesuai kebutuhan)
extensions = [".pdf", ".docx", ".pptx", ".xlsx"]

# Proses semua file dalam folder
success, failed = 0, []


# 2. Convert PDF -> .md

In [20]:
for file_path in sorted(input_folder.iterdir()):
    if file_path.suffix.lower() not in extensions:
        continue
    
    output_path = output_folder / (file_path.stem + ".md")
    
    if file_path.name == 'Pedoman tulisan hanya gambar.pdf':
        try:
            result = md_gambar.convert(str(file_path))
            output_path.write_text(result.text_content, encoding="utf-8")
            print(f"✓ {file_path.name} → {output_path.name}")
            success += 1
        except Exception as e:
            print(f"✗ {file_path.name}: {e}")
            failed.append(file_path.name)
    else:
        try:
            result = md_converter.convert(str(file_path))
            output_path.write_text(result.text_content, encoding="utf-8")
            print(f"✓ {file_path.name} → {output_path.name}")
            success += 1
        except Exception as e:
            print(f"✗ {file_path.name}: {e}")
            failed.append(file_path.name)

# Ringkasan
print(f"\nSelesai: {success} berhasil, {len(failed)} gagal")
if failed:
    print("Gagal:", ", ".join(failed))

✓ Pedoman skripsi tambahan.pdf → Pedoman skripsi tambahan.md
✓ Pedoman tulisan hanya gambar.pdf → Pedoman tulisan hanya gambar.md
✓ Pedoman tulisan tanpa gambar.pdf → Pedoman tulisan tanpa gambar.md
✓ Prosedur Tugas Akhir Prodi S-1 Ilmu Komputer.pdf → Prosedur Tugas Akhir Prodi S-1 Ilmu Komputer.md

Selesai: 4 berhasil, 0 gagal
